## Tennis Versus (06 - switching to XGBoost)

Adding ELO improved accuracy. Time to see if using XGBoost will improve it even further.

In [1]:
# data analytics + math
import pandas as pd 
import numpy as np 

# data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# other stuff used for later (need sci-kit learn too)
import xgboost # just installed it using !pip install

# set some parameters for pandas to view the data a bit better
pd.set_option('display.max_columns', 50) # see ALL columns
pd.set_option('display.width', 150)

# data = pd.read_parquet('elo_data.parquet')
data = pd.read_parquet('restructured_matches_02done.parquet')

In [2]:
data.tail()

,tourney_date,age_a,age_b,label,rank_diff,form_a,form_b,surface_form_a,surface_form_b,bp_pressure_a,bp_pressure_b,elo_a,elo_b,surface_elo_a,surface_elo_b,elo_diff,surface_elo_diff,surface_Clay,surface_Grass,surface_Hard,h2h_winrate_a
50415,2026-06-20,29.092,27.984,1,-5.0,0.8,0.7,0.8,0.8,7.1,5.2,1930.772404,1760.886904,1756.968148,1624.486873,169.885500,132.481275,0,1,0,1.000000
50416,2026-06-21,28.646,28.416,0,-17.0,0.7,0.8,0.8,0.8,4.2,6.2,1896.856222,1900.577897,1910.009123,1699.280877,-3.721675,210.728246,0,1,0,1.000000
50417,2026-06-21,27.855,29.095,1,-1.0,0.7,0.8,0.4,0.4,7.9,5.8,1853.299843,1939.518030,1596.398377,1767.146610,-86.218186,-170.748233,0,1,0,0.714286
50418,2026-06-21,28.416,28.646,1,17.0,0.8,0.7,0.7,0.7,6.2,4.2,1916.406514,1881.027605,1723.947694,1885.342306,35.378909,-161.394611,0,1,0,0.000000
50419,2026-06-21,29.095,27.855,0,1.0,0.8,0.7,0.8,0.8,5.8,7.9,1919.627079,1873.190794,1743.860709,1619.684279,46.436286,124.176430,0,1,0,0.285714


### no scaling required
 for XGBoost, there is no scaling required.

In [95]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

train = data[
    (data['tourney_date'] >= '2023-01-01') &
    (data['tourney_date'] < '2026-01-01')
]

test = data[
    (data['tourney_date'] >= '2026-01-01')
    # (data['tourney_date'] < '2026-01-01')
]

# all the 'dependent' variables
features = data.columns.tolist()
features.remove('tourney_date')
features.remove('label')

# testing stuff
# features.remove('age_a')
# features.remove('age_b')
# features.remove('bp_pressure_a')
# features.remove('bp_pressure_b')
# features.remove('form_a')
# features.remove('form_b')
features.remove('h2h_winrate_a')
features.remove('rank_diff')

X_train = train[features]
y_train = train['label']

X_test = test[features]
y_test = test['label']

# verifying shape
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

print(X_train.info())

(17152, 17)
(17152,)
(3005, 17)
(3005,)
<class 'pandas.core.frame.DataFrame'>
Index: 17152 entries, 30244 to 47403
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age_a             17152 non-null  float64
 1   age_b             17152 non-null  float64
 2   form_a            17152 non-null  float64
 3   form_b            17152 non-null  float64
 4   surface_form_a    17152 non-null  float64
 5   surface_form_b    17152 non-null  float64
 6   bp_pressure_a     17152 non-null  float64
 7   bp_pressure_b     17152 non-null  float64
 8   elo_a             17152 non-null  float64
 9   elo_b             17152 non-null  float64
 10  surface_elo_a     17152 non-null  float64
 11  surface_elo_b     17152 non-null  float64
 12  elo_diff          17152 non-null  float64
 13  surface_elo_diff  17152 non-null  float64
 14  surface_Clay      17152 non-null  int64  
 15  surface_Grass     17152 non-null  int64  
 16  s

### train the model

using ```XGBClassifier()``` from xgboost

In [96]:
model = XGBClassifier(
    n_estimators=100,       # number of trees
    max_depth=4,            # how deep each tree can grow
    learning_rate=0.1,      # how much each tree contributes
    eval_metric='logloss',  # what to optimise internally
    random_state=42
)
# model = XGBClassifier(
#     n_estimators=500,
#     max_depth=3,
#     learning_rate=0.01,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     eval_metric='logloss',
#     random_state=42
# )
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [99]:
# predictions
y_pred = model.predict(X_test) # will give a hard 0 or 1
y_pred_proba = model.predict_proba(X_test)[:, 1] # will give a probability instead ( returns [failing, passing] we want index 1)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
logloss = log_loss(y_test, y_pred_proba)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Log loss:  {logloss:.3f}")
print(f"AUC-ROC:   {auc:.3f}")

# naive = (X_test['rank_diff'] < 0).astype(int)
# naive_accuracy = accuracy_score(y_test, naive)

# print(f"Naive accuracy:  {naive_accuracy:.3f}") # higher....

Accuracy:  0.702
Log loss:  0.579
AUC-ROC:   0.765


### saving it

let's use it for Wimbledon now


In [101]:
import pickle

pickle.dump(model, open('tennis_model_xgb.pkl', 'wb'))